# Remaining useful life prediction with C-MAPSS

RUL prediction estimates the remaining cycles at each observation. C-MAPSS
training engines run to failure, while the test set ends before failure and
provides the true RUL at each engine's final observation.

## Learning goals

- construct row-level RUL without crossing engine boundaries;
- split training data by engine;
- evaluate final-observation predictions with the official test targets.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pdmdata").is_dir())
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

import pdmdata
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

In [ ]:
pdmdata.download("cmapss")
from pdmdata.cmapss import rul
train = pdmdata.load("cmapss", subset="FD001", split="train", with_rul=True)
test = pdmdata.load("cmapss", subset="FD001", split="test")
test_rul = rul("FD001")
train.select("unit_number", "cycle", "RUL").head()

In [ ]:
units = train["unit_number"].unique().sort().to_numpy()
development_units = units[: int(len(units) * 0.8)]
validation_units = units[int(len(units) * 0.8) :]
features = [
    column for column in train.columns
    if column.startswith(("operation_", "sensor_"))
]

development = train.filter(pl.col("unit_number").is_in(development_units))
validation = train.filter(pl.col("unit_number").is_in(validation_units))
model = HistGradientBoostingRegressor(loss="absolute_error", random_state=0)
model.fit(development.select(features).to_numpy(), development["RUL"].to_numpy())
validation_prediction = model.predict(validation.select(features).to_numpy())
print(
    "Validation MAE:",
    f"{mean_absolute_error(validation['RUL'].to_numpy(), validation_prediction):.1f} cycles",
)

In [ ]:
model.fit(train.select(features).to_numpy(), train["RUL"].to_numpy())
last_test_rows = test.sort("cycle").group_by("unit_number").last().sort("unit_number")
prediction = np.maximum(model.predict(last_test_rows.select(features).to_numpy()), 0)
actual = last_test_rows.select("unit_number").join(
    test_rul, on="unit_number", validate="1:1", maintain_order="left"
)["RUL"].to_numpy()
print(f"Official test MAE: {mean_absolute_error(actual, prediction):.1f} cycles")
print(f"Official test RMSE: {root_mean_squared_error(actual, prediction):.1f} cycles")

## Interpretation and limitations

Uncapped RUL counts the cycles remaining to failure; it does not identify fault
onset or assume that degradation began at the first cycle.
Common C-MAPSS studies use a piecewise cap and specialized asymmetric scores;
those choices must be reported because they materially change the task.